# CIFAR-10 speedrun trên GPU 4GB

Train một model đạt **≥ 94% test accuracy trên CIFAR-10** nhanh nhất có thể trên RTX 3050 Laptop 4GB, với peak
memory **≤ 1.5 GB** (GPU dùng chung). Đây là bài gom cả module: pipeline dữ liệu không để GPU chờ (mục 3), mixed
precision và `channels_last` (mục 4), đo bộ nhớ và profiler (mục 5), residual + BatchNorm (mục 6), và kỷ luật thí
nghiệm (ablation có seed). Kết quả và `REPORT.md` là một bài portfolio.

**Cách làm bài**
- Hàm dưới đây là **API test gọi** (tên và thứ tự tham số giữ nguyên; được thêm tham số keyword có mặc định). Phần
  còn lại của thiết kế là của bạn.
- Cell gắn tag `explore` chạy thí nghiệm thật; test bỏ qua chúng. Chọn kernel `ai-gpu` (xem `setup/gpu/README.md`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/02-pytorch-deep-dive/exercises/advanced/01-cifar10-speedrun -q`

In [ ]:
# explore: giới hạn luồng CPU, tìm gốc repo, kiểm CIFAR-10 (137 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import torch; GPU dùng chung, CPU cũng vậy
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODULE = ROOT / "tracks/ai-engineer/modules/02-pytorch-deep-dive"
CIFAR = ROOT / "data/raw/cifar10"
if not (CIFAR.is_dir() and any(CIFAR.iterdir())):
    # kernel ai-gpu không có thư viện tải dữ liệu: khi đó chạy ở terminal `uv run python data/download_dataset.py cifar10`
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("cifar10")
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import math
import time

import torch
import torch.nn.functional as F
from torch import nn

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## API

In [ ]:
def build_model(seed: int, width: int = 64) -> nn.Module:
    """Model cho ảnh (N, 3, 32, 32) đã chuẩn hoá, trả logits (N, 10). Mọi khởi tạo lấy từ `seed`.

    Gợi ý: ResNet-9 (mục 4 lý thuyết, bài Intermediate 01) với độ rộng `width`. Bạn được đổi kiến trúc.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def lr_at(step: int, total_steps: int, peak_lr: float, warmup_frac: float) -> float:
    """Lịch lr tam giác theo BƯỚC: tăng tuyến tính từ 0 lên peak_lr trong
    warm = max(1, round(total_steps * warmup_frac)) bước đầu (bước `warm` đạt đúng peak_lr),
    rồi giảm tuyến tính về 0 ở bước total_steps (không âm sau đó)."""
    # TODO
    raise NotImplementedError

In [ ]:
def predict_tta(model: nn.Module, x: torch.Tensor, batch_size: int = 1000) -> torch.Tensor:
    """Logits với test-time augmentation lật ngang: trung bình model(x) và model(x lật theo chiều rộng).

    x: float (N, 3, 32, 32) đã chuẩn hoá, ở CPU. Chạy theo batch trên device của model, ở eval mode và không
    dựng graph; trả tensor float32 (N, 10) ở CPU; trả model về đúng chế độ (train/eval) như trước khi gọi.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def fit(model: nn.Module, x_u8: torch.Tensor, y: torch.Tensor, epochs: int, batch_size: int = 512, **options) -> dict:
    """Train `model` trên ảnh uint8 (N, 3, 32, 32) và nhãn int64, trên DEVICE.

    Trả {"loss": [train loss trung bình mỗi epoch, float], "seconds": thời gian train (float)}.
    Chạy được trên CPU (test gọi với model nhỏ, 3 epoch, batch 64). `options` để bạn chạy ablation
    (vd. amp=False, channels_last=False, label_smoothing=0.0...).
    """
    # TODO
    raise NotImplementedError

## Thí nghiệm

In [ ]:
# explore: CIFAR-10 thật: 50 000 ảnh train, 10 000 ảnh test, uint8 (N, 3, 32, 32). Decode bằng data.py của bài Basic 03
import importlib.util

for path in [MODULE / "exercises/basic/03-fast-dataloader/starter/data.py", MODULE / "solutions/basic/03-fast-dataloader/data.py"]:
    spec = importlib.util.spec_from_file_location(f"basic03_{path.parent.name}", path)
    data_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(data_mod)
    try:
        x_train, y_train = data_mod.decode_cifar_parquet(CIFAR / "plain_text/train-00000-of-00001.parquet")
        x_test, y_test = data_mod.decode_cifar_parquet(CIFAR / "plain_text/test-00000-of-00001.parquet")
        break
    except NotImplementedError:
        continue
x_test_norm = (x_test.float() / 255 - MEAN) / STD
print(DEVICE, "| train", tuple(x_train.shape), "| test", tuple(x_test.shape))

In [ ]:
# explore: lần chạy chính. Đo thời gian train, peak memory, test accuracy (có và không TTA)
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
model = build_model(0)
hist = fit(model, x_train, y_train, epochs=16)          # TODO: chọn số epoch của bạn
logits = predict_tta(model, x_test_norm)
peak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else float("nan")
print(f"train {hist['seconds']:.0f} s | peak {peak:.0f} MiB | test acc TTA {(logits.argmax(1) == y_test).float().mean():.4f}")

In [ ]:
# explore: TODO ablation: mỗi lần bỏ MỘT thành phần (AMP, channels_last, dữ liệu trên GPU, label smoothing, TTA...),
# cùng số epoch, ít nhất 2 seed cho cấu hình chính. Ghi thời gian, peak memory, test acc vào một bảng.

In [ ]:
# explore: TODO torch.profiler cho vài bước của fit (bỏ bước warmup), xuất trace, đọc: GPU có chờ không, kernel nào lớn nhất